# 03 Data Cleaning

Notebook ini melakukan cleaning terhadap dataset gabungan berdasarkan hasil assessment sebelumnya.

## Langkah Cleaning

1. Setup dan load data
2. Handle missing values
3. Standardize date format
4. Clean amount values
5. Remove duplicates
6. Standardize text fields
7. Handle outliers
8. Data type conversion
9. Final validation
10. Generate cleaning report dan export dataset

## Setup Environment

Tujuan: menyiapkan library, path project, dataset raw gabungan, dan assessment report.

In [1]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
from datetime import datetime
import sys
from IPython.display import display

project_root = Path.cwd().resolve()
if project_root.name == 'notebooks':
    project_root = project_root.parent

src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.append(str(src_path))

try:
    from fingo_ds1.config import INTERIM_DATA_PATH, REPORTS_PATH
except ImportError:
    INTERIM_DATA_PATH = project_root / 'data' / 'interim'
    REPORTS_PATH = project_root / 'reports'

interim_data_path = Path(INTERIM_DATA_PATH)
reports_path = Path(REPORTS_PATH)
reports_path.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

df_raw = pd.read_csv(interim_data_path / 'combined_raw_transactions.csv', low_memory=False)

with open(interim_data_path / 'assessment_report.json', 'r', encoding='utf-8') as file:
    assessment = json.load(file)

print(f'Raw dataset loaded: {df_raw.shape}')
print('Assessment report loaded')
print(f'Identified issue categories: {len(assessment)}')
display(df_raw.head())

Raw dataset loaded: (119021, 66)
Assessment report loaded
Identified issue categories: 6


,date,amount,category,description,data_source,order_id,total_qty,total_weight_gr,total_returned_qty,total_diskon,product_categories,num_product_categories,status_pesanan,alasan_pembatalan,opsi_pengiriman,metode_pembayaran,kota_kabupaten,provinsi,ongkos_kirim_dibayar_oleh_pembeli,estimasi_potongan_biaya_pengiriman,total_pembayaran,perkiraan_ongkos_kirim,waktu_pesanan_dibuat,source_file,source_path,product_category,jumlah,returned_quantity,total_berat,waktu_pengiriman_diatur,status_pembatalan_pengembalian,no_resi,antar_ke_counter_pick_up,pesanan_harus_dikirimkan_sebelum_menghindari_keterlambatan,waktu_pembayaran_dilakukan,sku_induk,nomor_referensi_sku,nama_variasi,harga_awal,harga_setelah_diskon,total_harga_produk,diskon_dari_penjual,diskon_dari_shopee,berat_produk,jumlah_produk_di_pesan,voucher_ditanggung_penjual,cashback_koin,voucher_ditanggung_shopee,paket_diskon,paket_diskon_diskon_dari_shopee,paket_diskon_diskon_dari_penjual,potongan_koin_shopee,diskon_kartu_kredit,ongkos_kirim_pengembalian_barang,catatan_dari_pembeli,catatan,username_pembeli,nama_penerima,no_telepon,alamat_pengiriman,waktu_pesanan_selesai,mode,subcategory,income_expense,currency,type
0,2024-04-01 00:15:00,38300.0,Celengan,ORD_0000001,household_transaction,ORD_0000001,2.0,2000.0,0.0,0.0,Celengan,1.0,Selesai,NaN,Reguler (Cashless)-SPX Standard,Saldo ShopeePay,KOTA SERANG,BANTEN,0.0,10000.0,38300,10000.0,2024-04-01 00:15,AprilSales2024_clean.xlsx,data/raw/daily_household_transaction/daily_hou...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-04-01 01:47:00,18576.0,Celengan,ORD_0000002,household_transaction,ORD_0000002,1.0,500.0,0.0,0.0,Celengan,1.0,Selesai,NaN,Hemat Kargo-SPX Hemat,COD (Bayar di Tempat),KOTA SEMARANG,JAWA TENGAH,0.0,14500.0,18576,14500.0,2024-04-01 01:47,AprilSales2024_clean.xlsx,data/raw/daily_household_transaction/daily_hou...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2024-04-01 04:25:00,7069.0,Celengan,ORD_0000003,household_transaction,ORD_0000003,1.0,500.0,0.0,0.0,Celengan,1.0,Selesai,NaN,Hemat Kargo-SPX Hemat,SeaBank Bayar Instan,KAB. BOGOR,JAWA BARAT,0.0,8000.0,7069,8000.0,2024-04-01 04:25,AprilSales2024_clean.xlsx,data/raw/daily_household_transaction/daily_hou...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2024-04-01 04:41:00,32200.0,Mangkok Sambal / Saus,ORD_0000004,household_transaction,ORD_0000004,2.0,400.0,0.0,0.0,Mangkok Sambal / Saus,1.0,Selesai,NaN,Hemat Kargo-SPX Hemat,COD (Bayar di Tempat),KOTA JAMBI,JAMBI,0.0,20000.0,32200,20000.0,2024-04-01 04:41,AprilSales2024_clean.xlsx,data/raw/daily_household_transaction/daily_hou...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2024-04-01 06:12:00,0.0,"Keranjang, Other, Tempat Nasi",ORD_0000005,household_transaction,ORD_0000005,3.0,3600.0,0.0,0.0,"Keranjang, Other, Tempat Nasi",3.0,Batal,Dibatalkan oleh Pembeli. Alasan: Ubah Pesanan ...,Hemat Kargo-SPX Hemat,COD (Bayar di Tempat),KOTA TANGERANG,BANTEN,0.0,0.0,0,8000.0,2024-04-01 06:12,AprilSales2024_clean.xlsx,data/raw/daily_household_transaction/daily_hou...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Reusable Cleaning Helpers

Tujuan: menyiapkan fungsi logging, parsing, text cleaning, outlier capping, dan report generation.

In [2]:
cleaning_log = {
    'initial_shape': df_raw.shape,
    'steps': [],
}


def log_step(step_name, frame, removed_records=0, details=None):
    step = {
        'step': step_name,
        'records_remaining': int(len(frame)),
        'records_removed_step': int(removed_records),
        'records_removed_total': int(cleaning_log['initial_shape'][0] - len(frame)),
        'shape': [int(frame.shape[0]), int(frame.shape[1])],
    }

    if details:
        step['details'] = details

    cleaning_log['steps'].append(step)
    print(f'{step_name}: {len(frame):,} records remaining')


def parse_mixed_dates(series):
    values = series.astype('string').str.strip().replace({'': pd.NA, 'nan': pd.NA, 'None': pd.NA})

    try:
        return pd.to_datetime(values, errors='coerce', format='mixed', dayfirst=True)
    except TypeError:
        parsed = pd.to_datetime(values, errors='coerce')
        missing_mask = parsed.isna() & values.notna()
        parsed.loc[missing_mask] = pd.to_datetime(values[missing_mask], errors='coerce', dayfirst=True)
        return parsed


def parse_amount(series):
    values = series.astype(str).str.replace(r'[^0-9.\-]', '', regex=True)
    values = values.replace({'': np.nan, 'nan': np.nan, 'None': np.nan})
    return pd.to_numeric(values, errors='coerce')


def clean_text(series, fill_value=''):
    return (
        series.fillna(fill_value)
        .astype('string')
        .str.strip()
        .str.replace(r'\s+', ' ', regex=True)
    )


def clean_category(series):
    category_mapping = {
        'food': 'food_beverage',
        'foods': 'food_beverage',
        'food drink': 'food_beverage',
        'food  drink': 'food_beverage',
        'beverage': 'food_beverage',
        'drink': 'food_beverage',
        'transport': 'transportation',
        'transportation': 'transportation',
        'health': 'healthcare',
        'medical': 'healthcare',
        'entertainment': 'entertainment',
        'fun': 'entertainment',
        'cloth': 'clothing',
        'clothes': 'clothing',
        'fashion': 'clothing',
    }

    cleaned = (
        clean_text(series, 'unknown')
        .str.lower()
        .str.replace(r'[^\w\s]', '', regex=True)
        .str.replace(r'\s+', ' ', regex=True)
        .str.strip()
        .replace('', 'unknown')
    )
    return cleaned.replace(category_mapping)


def cap_outliers(series, percentile=0.99):
    cap_value = series.quantile(percentile)
    capped_count = int((series > cap_value).sum())
    return series.clip(upper=cap_value), cap_value, capped_count


def build_cleaning_report(summary):
    lines = [
        '=' * 80,
        'DATA CLEANING REPORT',
        '=' * 80,
        '',
        'BEFORE CLEANING',
        '-' * 80,
        f"Total Records: {summary['before']['records']:,}",
        f"Total Columns: {summary['before']['columns']}",
        f"Missing Values: {summary['before']['missing_values']:,}",
        f"Duplicates: {summary['before']['duplicates']:,}",
        '',
        'AFTER CLEANING',
        '-' * 80,
        f"Total Records: {summary['after']['records']:,}",
        f"Total Columns: {summary['after']['columns']}",
        f"Missing Values: {summary['after']['missing_values']:,}",
        f"Duplicates: {summary['after']['duplicates']:,}",
        '',
        'REMOVED DATA',
        '-' * 80,
        f"Total Removed: {summary['removed']['total_records']:,}",
        f"Percentage: {summary['removed']['percentage']}%",
        f"Retention Rate: {summary['removed']['retention_rate']}%",
        '',
        'CLEANING STEPS',
        '-' * 80,
    ]

    for step in summary['cleaning_steps']:
        lines.append(f"{step['step']}: {step['records_remaining']:,} records")

    lines.extend(['', '=' * 80])
    return '\n'.join(lines)

## Create Cleaning Tracker

Tujuan: membuat working dataframe dan mencatat jumlah record setelah setiap tahap cleaning.

In [3]:
df = df_raw.copy()
log_step('Initial dataset', df)
print(f'Initial records: {len(df):,}')

Initial dataset: 119,021 records remaining
Initial records: 119,021


## Handle Missing Values

Tujuan: menghapus missing pada kolom critical dan imputasi missing non-critical.

In [4]:
missing_before = pd.DataFrame(
    {
        'missing_count': df.isnull().sum(),
        'missing_pct': (df.isnull().sum() / len(df) * 100).round(2),
    }
).sort_values('missing_pct', ascending=False)

print('=== MISSING VALUES BEFORE CLEANING ===\n')
print(missing_before[missing_before['missing_count'] > 0])

before = len(df)
df = df[df['date'].notna() & df['amount'].notna()].copy()
log_step('Remove critical missing values', df, before - len(df))

category_missing = int(df['category'].isna().sum())
description_missing = int(df['description'].isna().sum())

df['category'] = df['category'].fillna('Unknown')
df['description'] = df['description'].fillna('')

log_step(
    'Impute non-critical missing values',
    df,
    0,
    {'category_imputed': category_missing, 'description_imputed': description_missing},
)

=== MISSING VALUES BEFORE CLEANING ===

                                                    missing_count  missing_pct
catatan                                                    119021       100.00
nomor_referensi_sku                                        119021       100.00
sku_induk                                                  119021       100.00
status_pembatalan_pengembalian                             119001        99.98
catatan_dari_pembeli                                       118905        99.90
nama_variasi                                               117567        98.78
type                                                       117521        98.74
subcategory                                                117195        98.47
income_expense                                             116560        97.93
mode                                                       116560        97.93
currency                                                   116560        97.93
waktu_pesana

Remove critical missing values: 107,937 records remaining
Impute non-critical missing values: 107,937 records remaining


## Standardize Date Format

Tujuan: parse tanggal, hapus tanggal invalid/future, dan filter periode valid 2020-2024.

In [5]:
df['date_clean'] = parse_mixed_dates(df['date'])

unparseable_dates = int(df['date_clean'].isna().sum())
print('=== DATE STANDARDIZATION ===\n')
print(f'Unparseable dates: {unparseable_dates:,}')

before = len(df)
df = df[df['date_clean'].notna()].copy()
log_step('Remove invalid dates', df, before - len(df))

future_mask = df['date_clean'] > datetime.now()
future_count = int(future_mask.sum())
df = df[~future_mask].copy()
log_step('Remove future dates', df, future_count)

min_date = pd.Timestamp('2020-01-01')
max_date = pd.Timestamp('2024-12-31')
before = len(df)
df = df[(df['date_clean'] >= min_date) & (df['date_clean'] <= max_date)].copy()
log_step('Filter date range 2020-2024', df, before - len(df))

print(f"Final date range: {df['date_clean'].min()} to {df['date_clean'].max()}")

=== DATE STANDARDIZATION ===

Unparseable dates: 0


Remove invalid dates: 107,937 records remaining
Remove future dates: 107,937 records remaining


Filter date range 2020-2024: 54,408 records remaining
Final date range: 2020-01-02 00:00:00 to 2024-12-29 00:00:00


## Clean Amount Values

Tujuan: parse amount ke numeric, hapus amount invalid/negatif, pertahankan zero amount, dan cap outlier ekstrem.

In [6]:
df['amount_clean'] = parse_amount(df['amount'])

print('=== AMOUNT CLEANING ===\n')
print(f"Non-numeric amounts: {df['amount_clean'].isna().sum():,}")

before = len(df)
df = df[df['amount_clean'].notna()].copy()
log_step('Remove non-numeric amounts', df, before - len(df))

negative_mask = df['amount_clean'] < 0
negative_count = int(negative_mask.sum())
df = df[~negative_mask].copy()
log_step('Remove negative amounts', df, negative_count)

zero_count = int((df['amount_clean'] == 0).sum())
print(f'Zero amounts kept: {zero_count:,} ({zero_count / len(df) * 100:.2f}%)')

positive_median = float(df.loc[df['amount_clean'] > 0, 'amount_clean'].median())
extreme_threshold = positive_median * 10
extreme_count = int((df['amount_clean'] > extreme_threshold).sum())

df['amount_clean'], cap_value, capped_count = cap_outliers(df['amount_clean'], 0.99)

log_step(
    'Cap extreme outliers',
    df,
    0,
    {
        'positive_median': positive_median,
        'extreme_threshold_10x_median': extreme_threshold,
        'extreme_values_detected': extreme_count,
        'cap_percentile': 0.99,
        'cap_value': float(cap_value),
        'capped_count': capped_count,
    },
)

print(f'Extreme values detected (>10x median): {extreme_count:,}')
print(f'Capped {capped_count:,} values at 99th percentile: {cap_value:,.2f}')

=== AMOUNT CLEANING ===

Non-numeric amounts: 0
Remove non-numeric amounts: 54,408 records remaining
Remove negative amounts: 54,408 records remaining
Zero amounts kept: 7,688 (14.13%)


Cap extreme outliers: 54,408 records remaining
Extreme values detected (>10x median): 3,462
Capped 543 values at 99th percentile: 366,800.00


## Standardize Text Fields

Tujuan: membersihkan category dan description agar konsisten untuk duplicate removal dan analisis lanjut.

In [7]:
print('=== TEXT FIELD STANDARDIZATION ===\n')

unique_categories_before = int(df['category'].nunique())
df['category_clean'] = clean_category(df['category'])
df['description_clean'] = clean_text(df['description'])
unique_categories_after = int(df['category_clean'].nunique())

log_step(
    'Standardize text fields',
    df,
    0,
    {
        'unique_categories_before': unique_categories_before,
        'unique_categories_after': unique_categories_after,
    },
)

print(f'Unique categories before: {unique_categories_before}')
print(f'Unique categories after: {unique_categories_after}')
print('\nTop categories:')
print(df['category_clean'].value_counts().head(15))

=== TEXT FIELD STANDARDIZATION ===



Standardize text fields: 54,408 records remaining
Unique categories before: 449
Unique categories after: 449

Top categories:
category_clean
celengan                 16934
mangkok sambal saus       8664
aksesoris pintu           3556
baskom mangkok besar      2480
rak rak serbaguna         1873
nampan tray               1699
lunch box rantang         1688
other                     1658
tempat nasi               1346
keranjang                 1269
saringan strainer         1080
seal baut roof            1031
botol gelas mug            872
peralatan kamar mandi      818
peralatan makan            813
Name: count, dtype: Int64


## Remove Duplicates

Tujuan: menghapus exact duplicate dan duplicate berdasarkan key cleaned `date`, `amount`, `category`.

In [8]:
print('=== DUPLICATE REMOVAL ===\n')

exact_duplicates = int(df.duplicated().sum())
before = len(df)
df = df.drop_duplicates(keep='first').copy()
log_step('Remove exact duplicates', df, before - len(df), {'exact_duplicates_detected': exact_duplicates})

subset_columns = ['date_clean', 'amount_clean', 'category_clean']
subset_duplicates = int(df.duplicated(subset=subset_columns).sum())
before = len(df)
df = df.drop_duplicates(subset=subset_columns, keep='first').copy()
log_step('Remove duplicate transaction keys', df, before - len(df), {'subset_duplicates_detected': subset_duplicates})

print(f'Exact duplicates detected: {exact_duplicates:,}')
print(f'Subset duplicates detected: {subset_duplicates:,}')

=== DUPLICATE REMOVAL ===



Remove exact duplicates: 51,948 records remaining
Remove duplicate transaction keys: 20,352 records remaining
Exact duplicates detected: 2,460
Subset duplicates detected: 31,596


## Data Type Conversion

Tujuan: membuat final schema dengan tipe data konsisten untuk dataset cleaned.

In [9]:
df['date'] = df['date_clean']
df['amount'] = df['amount_clean'].astype(float)
df['category'] = df['category_clean'].astype('string')
df['description'] = df['description_clean'].astype('string')

final_columns = ['date', 'amount', 'category', 'description', 'data_source']
df_clean = df[final_columns].copy()

df_clean['date'] = pd.to_datetime(df_clean['date'])
df_clean['amount'] = df_clean['amount'].astype(float)
df_clean['category'] = df_clean['category'].astype('string')
df_clean['description'] = df_clean['description'].astype('string')
df_clean['data_source'] = df_clean['data_source'].astype('string')

print('Final data types:')
print(df_clean.dtypes)

Final data types:
date           datetime64[ns]
amount                float64
category       string[python]
description    string[python]
data_source    string[python]
dtype: object


## Final Validation

Tujuan: memastikan hasil cleaning tidak punya critical missing, invalid date, negative amount, atau duplicate key.

In [10]:
validation_metrics = {
    'final_shape': [int(df_clean.shape[0]), int(df_clean.shape[1])],
    'retention_rate': round(len(df_clean) / len(df_raw) * 100, 2),
    'missing_values': int(df_clean.isnull().sum().sum()),
    'invalid_dates': int(df_clean['date'].isnull().sum()),
    'negative_amounts': int((df_clean['amount'] < 0).sum()),
    'non_numeric_amounts': int(pd.to_numeric(df_clean['amount'], errors='coerce').isnull().sum()),
    'duplicates': int(df_clean.duplicated(subset=['date', 'amount', 'category']).sum()),
    'date_min': str(df_clean['date'].min()),
    'date_max': str(df_clean['date'].max()),
}

print('=== CLEANED DATASET VALIDATION ===\n')
print(json.dumps(validation_metrics, indent=2))
print('\nMissing values:')
print(df_clean.isnull().sum())
print('\nAmount statistics:')
print(df_clean['amount'].describe())

=== CLEANED DATASET VALIDATION ===

{
  "final_shape": [
    20352,
    5
  ],
  "retention_rate": 17.1,
  "missing_values": 0,
  "invalid_dates": 0,
  "negative_amounts": 0,
  "non_numeric_amounts": 0,
  "duplicates": 0,
  "date_min": "2020-01-02 00:00:00",
  "date_max": "2024-12-29 00:00:00"
}

Missing values:
date           0
amount         0
category       0
description    0
data_source    0
dtype: int64

Amount statistics:
count     20352.000000
mean      21201.861421
std       49261.431255
min           0.000000
25%          26.145500
50%         140.582500
75%       24000.000000
max      366800.000000
Name: amount, dtype: float64


## Data Distribution Check

Tujuan: mengecek distribusi hasil cleaning berdasarkan source, category, dan bulan.

In [11]:
print('=== DATA DISTRIBUTION ===\n')
print('Records per data source:')
print(df_clean['data_source'].value_counts())

print('\nRecords per category:')
print(df_clean['category'].value_counts().head(15))

print('\nMonthly distribution:')
monthly_distribution = df_clean.assign(year_month=df_clean['date'].dt.to_period('M'))['year_month'].value_counts().sort_index()
print(monthly_distribution.head(12))

=== DATA DISTRIBUTION ===

Records per data source:
data_source
household_transaction    18852
personal_finance          1500
Name: count, dtype: Int64

Records per category:
category
celengan                 5951
mangkok sambal saus      3109
aksesoris pintu          1321
baskom mangkok besar      843
other                     702
rak rak serbaguna         634
nampan tray               607
lunch box rantang         571
tempat nasi               480
keranjang                 441
saringan strainer         412
seal baut roof            373
peralatan makan           313
botol gelas mug           309
peralatan kamar mandi     300
Name: count, dtype: Int64

Monthly distribution:
year_month
2020-01    17
2020-02    22
2020-03    17
2020-04    17
2020-05    24
2020-06    27
2020-07    25
2020-08    31
2020-09    25
2020-10    29
2020-11    29
2020-12    27
Freq: M, Name: count, dtype: int64


## Generate Cleaning Report

Tujuan: menyusun summary before/after, retention, step log, dan menyimpan report text.

In [12]:
cleaning_summary = {
    'before': {
        'records': int(len(df_raw)),
        'columns': int(len(df_raw.columns)),
        'missing_values': int(df_raw.isnull().sum().sum()),
        'duplicates': int(df_raw.duplicated(subset=['date', 'amount']).sum()),
    },
    'after': {
        'records': int(len(df_clean)),
        'columns': int(len(df_clean.columns)),
        'missing_values': int(df_clean.isnull().sum().sum()),
        'duplicates': int(df_clean.duplicated(subset=['date', 'amount', 'category']).sum()),
    },
    'removed': {
        'total_records': int(len(df_raw) - len(df_clean)),
        'percentage': round((len(df_raw) - len(df_clean)) / len(df_raw) * 100, 2),
        'retention_rate': round(len(df_clean) / len(df_raw) * 100, 2),
    },
    'validation': validation_metrics,
    'assessment_reference': assessment,
    'cleaning_steps': cleaning_log['steps'],
}

report_text = build_cleaning_report(cleaning_summary)
report_path = reports_path / '03_cleaning_report.txt'

with open(report_path, 'w', encoding='utf-8') as file:
    file.write(report_text)

print(report_text)
print(f'\nReport saved to: {report_path}')

DATA CLEANING REPORT

BEFORE CLEANING
--------------------------------------------------------------------------------
Total Records: 119,021
Total Columns: 66
Missing Values: 5,305,227
Duplicates: 78,357

AFTER CLEANING
--------------------------------------------------------------------------------
Total Records: 20,352
Total Columns: 5
Missing Values: 0
Duplicates: 0

REMOVED DATA
--------------------------------------------------------------------------------
Total Removed: 98,669
Percentage: 82.9%
Retention Rate: 17.1%

CLEANING STEPS
--------------------------------------------------------------------------------
Initial dataset: 119,021 records
Remove critical missing values: 107,937 records
Impute non-critical missing values: 107,937 records
Remove invalid dates: 107,937 records
Remove future dates: 107,937 records
Filter date range 2020-2024: 54,408 records
Remove non-numeric amounts: 54,408 records
Remove negative amounts: 54,408 records
Cap extreme outliers: 54,408 records
S

## Save Cleaned Dataset

Tujuan: export dataset cleaned ke folder interim sebagai input feature engineering.

In [13]:
output_path = interim_data_path / 'cleaned_transactions.csv'
df_clean.to_csv(output_path, index=False)

print(f'Cleaned dataset saved to: {output_path}')
print(f'File size: {output_path.stat().st_size / 1024 / 1024:.2f} MB')
print('\nFinal dataset preview:')
display(df_clean.head(10))

Cleaned dataset saved to: /home/umaygans/05_nayyara_submission_1/nayyara_capstone/data/interim/cleaned_transactions.csv
File size: 1.47 MB

Final dataset preview:


,date,amount,category,description,data_source
0,2024-04-01 00:15:00,38300.0,celengan,ORD_0000001,household_transaction
1,2024-04-01 01:47:00,18576.0,celengan,ORD_0000002,household_transaction
2,2024-04-01 04:25:00,7069.0,celengan,ORD_0000003,household_transaction
3,2024-04-01 04:41:00,32200.0,mangkok sambal saus,ORD_0000004,household_transaction
4,2024-04-01 06:12:00,0.0,keranjang other tempat nasi,ORD_0000005,household_transaction
5,2024-04-01 07:09:00,40996.0,celengan,ORD_0000006,household_transaction
6,2024-04-01 07:23:00,0.0,nampan tray,ORD_0000007,household_transaction
7,2024-04-01 07:33:00,18280.0,celengan,ORD_0000008,household_transaction
8,2024-04-01 07:56:00,0.0,celengan,ORD_0000009,household_transaction
9,2024-04-01 08:27:00,92800.0,aksesoris mandi celengan keranjang,ORD_0000010,household_transaction


## Summary

### Cleaning Achievements

- Critical missing values removed
- Date format standardized and filtered to 2020-2024
- Amount values converted to numeric and capped at 99th percentile
- Duplicate transaction keys removed
- Category and description standardized
- Cleaning report generated
- Cleaned dataset exported

### Next Steps

- Proceed to feature engineering
- Create temporal features
- Calculate spending patterns
- Prepare for labeling